In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from analysis.src.data import complete_case_mask
from analysis.src.io import experiment_paths, load_data
from analysis.src.constants import COMPONENTS, COVARIATES, OUTCOMES

In [ ]:
OUT = experiment_paths('robustness/index_definition')
DISEASES = OUTCOMES

data = load_data()
data = data.loc[data[COMPONENTS].notna().all(axis = 1)].copy() # Require all five components

In [ ]:
# Create leave-one-component-out indices
scaler = StandardScaler()

loo_cols = []
for comp in COMPONENTS:
    other_comps = [c for c in COMPONENTS if c != comp]
    
    loo_sum = data[other_comps].sum(axis = 1)
    loo_name = f'walkability_no_{comp.split("_")[0]}'
    data[loo_name] = scaler.fit_transform(loo_sum.values.reshape(-1, 1)).flatten()
    
    loo_cols.append(loo_name)

print('Created LOO indices:', loo_cols)

In [ ]:
# PCA-weighted index
pca = PCA(n_components = 1)
pca_raw = pca.fit_transform(data[COMPONENTS])

if stats.pearsonr(pca_raw.flatten(), data['walkability_score'])[0] < 0:
    pca_raw = pca_raw * -1

data['walkability_pca'] = scaler.fit_transform(pca_raw).flatten()

print(f'PCA Variance Explained: {pca.explained_variance_ratio_[0] * 100:.2f}%')

In [ ]:
# Compare rank stability using Spearman correlations
all_indices = ['walkability_score', 'walkability_pca'] + loo_cols
corr_matrix = data[all_indices].corr(method = 'spearman')

display(corr_matrix[['walkability_score']].sort_values(by = 'walkability_score', ascending = False))

In [ ]:
# Fit regression sensitivity models
results = []
for outcome in DISEASES:
    model_data = data.loc[complete_case_mask(data, outcome, COVARIATES)]

    for index_name in all_indices:
        formula = f'np.log({outcome}) ~ {index_name} + ' + ' + '.join(COVARIATES)
        model = smf.glm(formula = formula, data = model_data, family = sm.families.Gaussian()).fit()
        ci = model.conf_int().loc[index_name]
        coefficient = model.params[index_name]
        
        results.append(
            {
                'outcome': outcome,
                'index_definition': index_name,
                'n_lsoas': int(model.nobs),
                'coefficient': coefficient,
                'ci_lower': ci.iloc[0],
                'ci_upper': ci.iloc[1],
                'p_value': model.pvalues[index_name],
                'effect_percent': (np.exp(coefficient) - 1) * 100
            }
        )

index_sensitivity_results = pd.DataFrame(results)
index_sensitivity_results.to_csv(OUT.index_sensitivity_csv, index = False)

display(index_sensitivity_results.round(4))

In [ ]:
corr_matrix.rename_axis('index_definition').reset_index().to_csv(OUT.rank_correlations_csv, index = False)

pd.DataFrame({'index_definition': all_indices, 'sd': [data[c].std(ddof = 1) for c in all_indices]}).to_csv(
    OUT.index_scales_csv, index = False
)